In [1]:
import torch
import requests
from time import perf_counter_ns # integer value of time in nanoseconds


In [2]:
def get_url_content(url):
    """Download content from a given URL"""
    response=requests.get(url, headers={'User-Agent':""})
    if response.status_code!=200: raise RuntimeError(f'Failed to download video. {response.status_code=}.')
    return response.content
    
def bench(f, average_over=10, warmup=2):
    """Function to time the execution of the given f function"""
    for _ in range(warmup): f()

    times=[]
    for _ in range(average_over):
        start=perf_counter_ns()
        f()
        end=perf_counter_ns()
        times.append(end-start)

    times=torch.tensor(times)*1e-6 # nanosecond to milisecond
    std=times.std().item()
    med=times.median().item()
    print(f'{med=:.2f} +- {std:.2f} ms')

In [3]:
from torchcodec.decoders import VideoDecoder

nasa_url = "https://download.pytorch.org/torchaudio/tutorial-assets/stream-api/NASAs_Most_Scientifically_Complex_Space_Observatory_Requires_Precision-MP4.mp4"
pre_downloaded_raw_video_bytes=get_url_content(nasa_url)
decoder=VideoDecoder(pre_downloaded_raw_video_bytes)
print(f'Video size in MB: {len(pre_downloaded_raw_video_bytes) //1024 // 1024}')
print(decoder.metadata)

Video size in MB: 253
VideoStreamMetadata:
  duration_seconds_from_header: 206.039167
  begin_stream_seconds_from_header: 0.0
  bit_rate: 9958354.0
  codec: h264
  stream_index: 0
  duration_seconds: 206.039167
  begin_stream_seconds: 0.0
  begin_stream_seconds_from_content: 0.0
  end_stream_seconds_from_content: 206.039167
  width: 1920
  height: 1080
  num_frames_from_header: 6175
  num_frames_from_content: 6175
  average_fps_from_header: 29.97003
  pixel_aspect_ratio: 1
  end_stream_seconds: 206.039167
  num_frames: 6175
  average_fps: 29.97003



In [4]:
def decode_from_existing_download():
    decoder=VideoDecoder(source=pre_downloaded_raw_video_bytes, seek_mode='approximate')
    return decoder[0]
def download_before_decode():
    raw_video_bytes=get_url_content(nasa_url)
    decoder=VideoDecoder(source=raw_video_bytes, seek_mode='approximate')
    return decoder[0]
def direct_url_to_ffmpeg():
    decoder=VideoDecoder(source=nasa_url, seek_mode='approximate')
    return decoder[0]

In [6]:
print("Decode from existing download: ")
bench(decode_from_existing_download)
print()

print('Download before decode:')
bench(download_before_decode)
print()

print('Direct url to FFmpeg:')
bench(direct_url_to_ffmpeg)
print()

Decode from existing download: 
med=172.90 +- 2.70 ms

Download before decode:
med=9996.87 +- 851.69 ms

Direct url to FFmpeg:
med=2179.63 +- 96.08 ms



In [5]:
import fsspec

def stream_while_decode():
    # The `client_kwargs` are passed down to the aiohttp module's client
    # session; we need to indicate that we need to trust the environment settings
    # for proxy configuration. Depending on your environment, you may not need this setting
    with fsspec.open(nasa_url, client_kwargs={'trust_env':True}) as file_like:
        decoder=VideoDecoder(file_like, seek_mode='approximate')
        return decoder[0]

print('Stream while decode:')
bench(stream_while_decode)

Stream while decode:
med=442.14 +- 23.61 ms


In [6]:
from pathlib import Path
import tempfile

# Create a local file to interact with
temp_dir=tempfile.mkdtemp()
nasa_video_path=Path(temp_dir)/'nasa_video.mp4'
with open(nasa_video_path, 'wb') as f: f.write(pre_downloaded_raw_video_bytes)

# A file-like class that is backed by an actual file, but it interscepts reads, and seeks to maintain counts
class FileOpCounter:
    def __init__(self, file):
        self._file=file
        self.num_reads=0
        self.num_seeks=0
    def read(self, size:int)->bytes:
        self.num_reads+=1
        return self._file.read(size)
    def seek(self, offset:int, whence:int)->int:
        self.num_seeks+=1
        return self._file.seek(offset, whence)

# Let's now get a file-like object from our class defined above, providing it a refernce to the file
# we created. We pass our file-like object to the decoder rather than the file itself
file_op_counter=FileOpCounter(open(nasa_video_path, 'rb'))
counter_decoder=VideoDecoder(file_op_counter, seek_mode='approximate')
print("Decoder initialization required "
     f"{file_op_counter.num_reads} reads and "
     f"{file_op_counter.num_seeks} seeks.")
init_reads=file_op_counter.num_reads
init_seeks=file_op_counter.num_seeks

first_frame=counter_decoder[0]
print('Decoder the first frame required '
     f"{file_op_counter.num_reads-init_reads} additional reads and "
     f"{file_op_counter.num_seeks-init_seeks} additional seeks")

Decoder initialization required 9 reads and 11 seeks.
Decoder the first frame required 2 additional reads and 1 additional seeks


In [7]:
def decode_from_existing_file_path():
    decoder=VideoDecoder(nasa_video_path, seek_mode='approximate')
    return decoder[0]
def decode_from_existing_open_file_object():
    with open(nasa_video_path, 'rb') as file:
        decoder=VideoDecoder(file, seek_mode='approximate')
        return decoder[0]
print("Decode from existing file path:")
bench(decode_from_existing_file_path)
print()

print('Decode from existing open file object:')
bench(decode_from_existing_open_file_object)

Decode from existing file path:
med=172.30 +- 7.33 ms

Decode from existing open file object:
med=176.18 +- 6.53 ms


In [11]:
# import shutil
# shutil.rmtree(temp_dir)